## Using an Individualized Baseline Approach to Predict Reinjury Among MLB Batters 

In [1]:
## Install pybaseball ##

%pip install pybaseball


Note: you may need to restart the kernel to use updated packages.


In [2]:
## Install seaborn ##

%pip install seaborn

## Install openpyxl ##

%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
## Load Relevant Packages ##

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pybaseball as pb

## Read in MLB Injuries Raw Data for 2022 Season ##

url = "https://raw.githubusercontent.com/abrown9008/Sports-Analytics-and-Intelligence-Lab/main/MLB%20Oblique%20Injuries/MLB%20Injuries%20Raw%20Data.xlsx"
injuries22 = pd.read_excel(url, engine="openpyxl",sheet_name='2022')

print(injuries22.head())

              Name                                                URL Team  \
0     Tommy Kahnle  https://www.fangraphs.com/players/tommy-kahnle...  LAD   
1   Caleb Ferguson  https://www.fangraphs.com/players/caleb-fergus...  LAD   
2       Nick Mears  https://www.fangraphs.com/players/nick-mears/2...  PIT   
3  Kyle Funkhouser  https://www.fangraphs.com/players/kyle-funkhou...  DET   
4  Blake Cederlind  https://www.fangraphs.com/players/blake-cederl...  PIT   

  Pos Injury / Surgery Date            Injury / Surgery     Status  \
0  RP   2020-08-04 00:00:00          Tommy John surgery  Activated   
1  RP   2020-09-23 00:00:00          Tommy John surgery  Activated   
2  RP   2021-02-09 00:00:00  Arthroscopic elbow surgery  Activated   
3  RP               Mar '22           Strained shoulder  60-Day IL   
4  RP   2021-03-23 00:00:00          Tommy John surgery  60-Day IL   

  IL Retro Date Eligible to Return Return Date            Latest Update  \
0    2022-04-04         2022-04-14 

In [2]:
## Pivot Data such that each row is a unique player ##

stint_cols = {
    'IL Retro Date': 'IL_Retro',
    'Return Date': 'Activate',
    'Injury / Surgery': 'Injury',
}

df = injuries22.copy()
for c in ['IL Retro Date', 'Return Date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# Stint 1 = earliest; rows with no IL date sort last
df = df.sort_values(['key_fangraphs', 'IL Retro Date'], na_position='last')
df['stint'] = df.groupby('key_fangraphs').cumcount() + 1

# Player-level attributes (first stint's value is kept if a player changed team)
players = df[['key_fangraphs', 'Name', 'Team', 'Pos', 'URL', 'sub_url']] \
    .drop_duplicates('key_fangraphs')

wide = (
    df.set_index(['key_fangraphs', 'stint'])[list(stint_cols)]
      .rename(columns=stint_cols)
      .unstack('stint')
)
wide.columns = [f'{col}_{n}' for col, n in wide.columns]

max_n = int(df['stint'].max())
ordered = [f'{c}_{n}' for n in range(1, max_n + 1) for c in stint_cols.values()]
wide = wide[ordered].reset_index()

injuries22_wide = players.merge(wide, on='key_fangraphs', how='left')

assert injuries22_wide['key_fangraphs'].is_unique
print(injuries22_wide.shape, df['stint'].value_counts().sort_index().to_dict())



(722, 24) {1: 722, 2: 210, 3: 42, 4: 5, 5: 1, 6: 1}


In [3]:
## Now let's subset to only non-pitchers and oblique injuries for the 2022 season ##

inj22_wide_batters_oblique = injuries22_wide[
    (injuries22_wide['Pos'] != 'P') &
    (injuries22_wide['Pos'] != 'SP') &
    (injuries22_wide['Pos'] != 'RP') &
    (injuries22_wide['Injury_1'].str.contains('oblique', case=False, na=False))
]

In [4]:
## Reusable: read a season sheet and pivot to one row per player ##

stint_cols = {
    'IL Retro Date': 'IL_Retro',
    'Return Date': 'Activate',
    'Injury / Surgery': 'Injury',
}
keep_cols = ['Name', 'URL', 'Team', 'Pos', 'IL Retro Date', 'Return Date',
             'Injury / Surgery', 'sub_url', 'key_fangraphs']

def build_wide(year):
    raw = pd.read_excel(url, engine="openpyxl", sheet_name=str(year))[keep_cols]

    # Recover missing keys from the FanGraphs URL (e.g. /players/name/19931/stats/batting)
    raw['key_fangraphs'] = (
        raw['key_fangraphs']
        .fillna(raw['URL'].str.extract(r'/players/[^/]+/([^/]+)/')[0])
        .astype(str)
    )

    df = raw.copy()
    for c in ['IL Retro Date', 'Return Date']:
        df[c] = pd.to_datetime(df[c], errors='coerce')

    df = df.sort_values(['key_fangraphs', 'IL Retro Date'], na_position='last')
    df['stint'] = df.groupby('key_fangraphs').cumcount() + 1

    players = df[['key_fangraphs', 'Name', 'Team', 'Pos', 'URL', 'sub_url']] \
        .drop_duplicates('key_fangraphs')

    wide = (
        df.set_index(['key_fangraphs', 'stint'])[list(stint_cols)]
          .rename(columns=stint_cols)
          .unstack('stint')
    )
    wide.columns = [f'{col}_{n}' for col, n in wide.columns]
    max_n = int(df['stint'].max())
    ordered = [f'{c}_{n}' for n in range(1, max_n + 1) for c in stint_cols.values()]
    wide = wide[ordered].reset_index()

    out = players.merge(wide, on='key_fangraphs', how='left')
    assert out['key_fangraphs'].is_unique
    assert out['key_fangraphs'].ne('nan').all(), f"{year}: unresolved keys"
    return out

## Build one wide dataframe per season ##

injuries_wide = {yr: build_wide(yr) for yr in [2022, 2023, 2024, 2025, 2026]}

injuries23_wide = injuries_wide[2023]
injuries24_wide = injuries_wide[2024]
injuries25_wide = injuries_wide[2025]
injuries26_wide = injuries_wide[2026]

for yr, d in injuries_wide.items():
    print(yr, d.shape)

2022 (722, 24)
2023 (677, 18)
2024 (619, 18)
2025 (634, 18)
2026 (675, 18)


In [5]:
## Row bind and add season column ##

all_wide = pd.concat(
    [d.assign(Season=yr) for yr, d in injuries_wide.items()],
    ignore_index=True
)

In [6]:
## Subset all_wide to only non-pitchers and oblique injuries for all seasons ##

all_wide_batters_oblique = all_wide[
    (all_wide['Pos'] != 'P') &
    (all_wide['Pos'] != 'SP') &
    (all_wide['Pos'] != 'RP') &
    (all_wide['Injury_1'].str.contains('oblique', case=False, na=False))
]

In [7]:
## Remove players with NA in Active_1 column ##

all_wide_batters_oblique = all_wide_batters_oblique[
    all_wide_batters_oblique['Activate_1'].notna()
]

In [9]:
## How many players whose first injury was an oblique injury experienced a second oblique injury? ##

print("Total Number of Players with Oblique Injuries:", all_wide_batters_oblique.shape[0])
print("Number of Players with a Second Injury:", all_wide_batters_oblique['Injury_2'].notna().sum())
print("Number of Players with a Second Oblique Injury:", all_wide_batters_oblique['Injury_2'].str.contains('oblique', case=False, na=False).sum())

Total Number of Players with Oblique Injuries: 138
Number of Players with a Second Injury: 35
Number of Players with a Second Oblique Injury: 5
